In [21]:
import trueq as tq
from trueq import Gate
import random
import numpy as np
import trueq.simulation as tqs

In [3]:
def random_tuple(n_qubits):

    # Generate a list of numbers from 0 to n
    numbers = list(range(n_qubits))
    
    # Shuffle the list to randomize the selection
    random.shuffle(numbers)
    
    # Determine the number of pairs (n//2 - 1)
    num_pairs = n_qubits // 2 
    
    # Create the required number of tuples
    tuple_list = [tuple(numbers[2 * i: 2 * i + 2]) for i in range(num_pairs)]
    
    # Find the remaining numbers that are not used in the tuples
    used_numbers = set(num for t in tuple_list for num in t)
    missing_numbers = list(set(range(n_qubits)) - used_numbers)

    return tuple_list, missing_numbers



def easy_cycle_NISQ(n_qubits):

    easy_gate_cycle = {}

    for i in range(n_qubits):
        easy_gate_cycle[i] = Gate.random(2)
    
    return tq.Cycle(easy_gate_cycle)


def hard_cycle_NISQ(n_qubits):

    hard_gate_cycle = {}
    list_tuples, missing_num = random_tuple(n_qubits)
    
    # 50/50 chance to remove one tuple
    if random.random() < 0.5 and list_tuples:
        list_tuples.pop(random.randint(0, len(list_tuples) - 1))

    for i in range(len(list_tuples)):
        hard_gate_cycle[list_tuples[i]] = Gate.cx


    return tq.Cycle(hard_gate_cycle, marker= 2)


def random_circuit_NISQ(n_qubits, n_hard_cycles):

    circ_lst = []

    for i in range(n_hard_cycles):

        circ_lst.append(easy_cycle_NISQ(n_qubits))
        circ_lst.append(hard_cycle_NISQ(n_qubits))
    
    circ_lst.append(easy_cycle_NISQ(n_qubits))

    final_circuit = tq.Circuit(circ_lst)
    final_circuit.measure_all()

    return final_circuit

In [10]:
random_circuit_NISQ(4,2).draw()

DisplayWrapper(<svg xmlns="http://w...)

In [34]:
def gate_replace(gate):
    if gate == Gate.cx:
        return gate
    else:
        return Gate.rp('X',3.6243/2)@gate
    
cnot_gate_rotation = 0.073675/2

sim_hard = tq.Simulator().add_gate_replace(gate_replace).add_overrotation(multi_sys=cnot_gate_rotation, match= tqs.GateMatch(Gate.cx))



In [35]:
gate_replace(Gate.i)

Gate(X)

In [19]:
cnot_fid= 0.9960947483409102
i_fid = 0.9997482357784453